# Data Lab 6 — Comparing Shifts & Trends Over Time
**Certificate 04 · Data Analytics**  |  [← Back to the Lab Hub](../../index.ipynb)

Uses a shared synthetic **factory production log** — one row per part made across 3 presses, 2 shifts, 5 days. pandas, NumPy and matplotlib are pre-installed in Colab, so there is nothing to install.

## Objectives
By the end you will be able to:
- Aggregate by shift, date and hour of day.
- Compare Day vs Night with a pivot table.
- Chart daily fail-rate trends and throughput.

## Load
Parse the timestamp column so we can work with time; add a numeric `fail` flag.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
DATA = "https://raw.githubusercontent.com/IF-JL/COEFAM-Labs/main/labs/cert04_data_analytics/data/"
df = pd.read_csv(DATA + "production_log.csv", parse_dates=["timestamp"])
df["fail"] = (df["result"] == "Fail").astype(int)
df.head()

## 1 · Day vs Night
One `groupby` gives a shift scorecard.

In [ ]:
by_shift = df.groupby("shift").agg(
    parts=("part_id", "size"),
    fail_rate=("fail", "mean"),
    avg_cycle=("cycle_time_s", "mean"),
    avg_temp=("oven_temp_c", "mean"),
).round(3)
print(by_shift)

## 2 · Shift x machine — a pivot table

In [ ]:
pivot = df.pivot_table(index="machine", columns="shift", values="fail", aggfunc="mean")
pivot.plot(kind="bar", figsize=(6,3))
plt.title("Fail rate by machine & shift"); plt.ylabel("fail rate"); plt.show()

## 3 · Daily trend
Does quality drift across the week?

In [ ]:
daily = df.groupby("date")["fail"].mean()
plt.figure(figsize=(7,3)); plt.plot(daily.index, daily.values, marker="o")
plt.title("Daily fail rate"); plt.ylabel("fail rate"); plt.xticks(rotation=20); plt.show()

## 4 · Throughput per day, per machine

In [ ]:
tp = df.groupby(["date", "machine"]).size().unstack()
tp.plot(figsize=(7,3), marker="o")
plt.title("Parts made per day"); plt.ylabel("parts"); plt.xticks(rotation=20); plt.show()

## 5 · Hour-of-day pattern
Extract the hour from the timestamp.

In [ ]:
df["hour"] = df["timestamp"].dt.hour
by_hour = df.groupby("hour")["fail"].mean()
plt.figure(figsize=(8,3)); plt.bar(by_hour.index, by_hour.values)
plt.title("Fail rate by hour of day"); plt.xlabel("hour"); plt.ylabel("fail rate"); plt.show()

## Debrief
- Is one shift clearly worse? By how much, and on which machine?
- Time is a feature: patterns by shift, day and hour often point straight at a cause.